# Prepare OpenBookQA for prompt optimization

This notebook loads the official AllenAI JSONL files, shows five examples, converts every record to a simple shared schema, removes exact duplicates from the training split, validates the splits, and writes JSONL plus combined JSON files under `data/processed/openbookqa/`.

In [8]:
import json
from pathlib import Path

In [9]:
def find_repo_root():
    """Find the repository root from the current notebook directory."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "AGENTS.md").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")


def read_jsonl(path):
    """Read non-empty JSON objects from a JSONL file."""
    with path.open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]


def show_examples(records, count=5):
    """Display a small number of records in readable JSON."""
    print(json.dumps(records[:count], ensure_ascii=False, indent=2))

In [10]:
REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data/openbookqa/original/OpenBookQA-V1-Sep2018/Data/Additional"
OUTPUT_DIR = REPO_ROOT / "data/processed/openbookqa"

RAW_SPLITS = {
    "train": RAW_DIR / "train_complete.jsonl",
    "validation": RAW_DIR / "dev_complete.jsonl",
    "test": RAW_DIR / "test_complete.jsonl",
}

raw_records = {split: read_jsonl(path) for split, path in RAW_SPLITS.items()}
for split, records in raw_records.items():
    print(f"{split}: {len(records):,} records")

train: 4,957 records
validation: 500 records
test: 500 records


## Five original examples

In [11]:
show_examples(raw_records["train"], count=5)

[
  {
    "id": "7-980",
    "question": {
      "stem": "The sun is responsible for",
      "choices": [
        {
          "text": "puppies learning new tricks",
          "label": "A"
        },
        {
          "text": "children growing up and getting old",
          "label": "B"
        },
        {
          "text": "flowers wilting in a vase",
          "label": "C"
        },
        {
          "text": "plants sprouting, blooming and wilting",
          "label": "D"
        }
      ]
    },
    "fact1": "the sun is the source of energy for physical cycles on Earth",
    "humanScore": "1.00",
    "clarity": "2.00",
    "turkIdAnonymized": "b356d338b7",
    "answerKey": "D"
  },
  {
    "id": "7-584",
    "question": {
      "stem": "When standing miles away from Mount Rushmore",
      "choices": [
        {
          "text": "the mountains seem very close",
          "label": "A"
        },
        {
          "text": "the mountains are boring",
          "label": "B"
     

In [12]:
def normalize_openbookqa(row, split):
    """Convert one AllenAI record to the prompt-optimization schema."""
    question = row["question"]
    choices = [
        {"label": choice["label"], "text": choice["text"]}
        for choice in question["choices"]
    ]
    choice_text = {choice["label"]: choice["text"] for choice in choices}
    answer = row["answerKey"]
    return {
        "id": row["id"],
        "dataset": "openbookqa",
        "task_type": "multiple_choice_qa",
        "split": split,
        "question": question["stem"],
        "choices": choices,
        "answer": answer,
        "answer_text": choice_text[answer],
        "fact": row.get("fact1", ""),
    }


def normalize_text(text):
    """Normalize text for reliable duplicate comparison."""
    return " ".join(text.split()).casefold()


def openbookqa_example_key(record):
    """Create a duplicate key from the question and ordered choices."""
    choices = tuple(
        (normalize_text(choice["label"]), normalize_text(choice["text"]))
        for choice in record["choices"]
    )
    return normalize_text(record["question"]), choices


def deduplicate_openbookqa(records):
    """Keep the first exact question-and-choices example and remove later copies."""
    unique_records = []
    first_record_by_key = {}
    for record in records:
        key = openbookqa_example_key(record)
        first_record = first_record_by_key.get(key)
        if first_record is not None:
            if first_record["answer"] != record["answer"]:
                raise ValueError(f"Conflicting answers for duplicate question: {record['question']}")
            continue
        first_record_by_key[key] = record
        unique_records.append(record)
    return unique_records, len(records) - len(unique_records)


def validate_openbookqa(records_by_split):
    """Check counts, IDs, duplicates, choices, and answers in all prepared splits."""
    expected_counts = {"train": 4952, "validation": 500, "test": 500}
    assert {key: len(value) for key, value in records_by_split.items()} == expected_counts
    all_records = [record for records in records_by_split.values() for record in records]
    assert len({record["id"] for record in all_records}) == len(all_records)
    train_keys = [openbookqa_example_key(record) for record in records_by_split["train"]]
    assert len(set(train_keys)) == len(train_keys)
    for record in all_records:
        assert len(record["choices"]) == 4
        labels = {choice["label"] for choice in record["choices"]}
        assert record["answer"] in labels
        assert record["question"].strip() and record["answer_text"].strip()


prepared = {
    split: [normalize_openbookqa(row, split) for row in rows]
    for split, rows in raw_records.items()
}
prepared["train"], removed_train_duplicates = deduplicate_openbookqa(prepared["train"])
assert removed_train_duplicates == 5
print(f"Removed {removed_train_duplicates} exact duplicates from train.")
validate_openbookqa(prepared)
print("Validation passed.")

Removed 5 exact duplicates from train.
Validation passed.


## Five prepared examples

In [13]:
show_examples(prepared["train"], count=5)

[
  {
    "id": "7-980",
    "dataset": "openbookqa",
    "task_type": "multiple_choice_qa",
    "split": "train",
    "question": "The sun is responsible for",
    "choices": [
      {
        "label": "A",
        "text": "puppies learning new tricks"
      },
      {
        "label": "B",
        "text": "children growing up and getting old"
      },
      {
        "label": "C",
        "text": "flowers wilting in a vase"
      },
      {
        "label": "D",
        "text": "plants sprouting, blooming and wilting"
      }
    ],
    "answer": "D",
    "answer_text": "plants sprouting, blooming and wilting",
    "fact": "the sun is the source of energy for physical cycles on Earth"
  },
  {
    "id": "7-584",
    "dataset": "openbookqa",
    "task_type": "multiple_choice_qa",
    "split": "train",
    "question": "When standing miles away from Mount Rushmore",
    "choices": [
      {
        "label": "A",
        "text": "the mountains seem very close"
      },
      {
        "l

In [14]:
def write_jsonl(path, records):
    """Write records as one JSON object per line."""
    with path.open("w", encoding="utf-8") as stream:
        for record in records:
            stream.write(json.dumps(record, ensure_ascii=False) + "\n")


def write_json(path, value):
    """Write a JSON value with readable indentation."""
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for split, records in prepared.items():
    write_jsonl(OUTPUT_DIR / f"{split}.jsonl", records)

all_records = [record for split in ("train", "validation", "test") for record in prepared[split]]
write_json(OUTPUT_DIR / "all.json", all_records)
write_json(
    OUTPUT_DIR / "dataset_info.json",
    {
        "dataset": "openbookqa",
        "task_type": "multiple_choice_qa",
        "splits": {split: len(records) for split, records in prepared.items()},
        "files": ["train.jsonl", "validation.jsonl", "test.jsonl", "all.json"],
    },
)
print(f"Saved {len(all_records):,} records to {OUTPUT_DIR}")

Saved 5,952 records to /storage2/home/aunabilchakma/codes/RE_Prompt_optimizer/data/processed/openbookqa
